# Environment Setup

This notebook walks through installing and verifying `oneccl_bindings_for_pytorch`
on a CRI inference node. Run this before any other notebook.

## Step 1 -- Load Intel oneAPI Modules

In [ ]:
# Check current module environment
import subprocess
result = subprocess.run("module list 2>&1", shell=True, capture_output=True, text=True)
print(result.stdout or result.stderr)

# If Intel MPI not loaded, uncomment:
# !module load intel/mpi intel/oneapi/pytorch

## Step 2 -- Install oneCCL PyTorch Bindings

In [ ]:
# Install from Intel's PyPI channel
# Pin to the version matching your PyTorch build
!pip install oneccl_bind_pt --extra-index-url https://pytorch-extension.intel.com/release-whl/stable/xpu/us/ 2>&1 | tail -5

## Step 3 -- Full Verification

In [ ]:
%%writefile /tmp/ccl_verify.py
import os
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_LOG_LEVEL"] = "warn"

dist.init_process_group(backend="ccl")

rank = dist.get_rank()
size = dist.get_world_size()
device = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

t = torch.tensor([float(rank)], device=device)
dist.all_reduce(t)
expected = sum(range(size))

status = "PASS" if abs(t.item() - expected) < 0.01 else "FAIL"
print(f"[rank {rank}/{size}] device={device}  allreduce={t.item():.0f}  expected={expected}  {status}")

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_verify.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    print("\nTroubleshooting:")
    print("  - Ensure `module load intel/mpi` was run before launching Jupyter")
    print("  - Check: echo $I_MPI_ROOT")
    print("  - Check: python -c 'import oneccl_bindings_for_pytorch'")

## Step 4 -- NUMA Topology Inspection

In [ ]:
# This is the most important pre-flight check for CRI nodes
# You want GPUs distributed evenly across NUMA domains

import subprocess

print("=== NUMA Hardware ===")
r = subprocess.run("numactl --hardware", shell=True, capture_output=True, text=True)
print(r.stdout)

print("=== XPU Devices ===")
r2 = subprocess.run(
    "python -c \""
    "import torch; "
    "[print(f'xpu:{i} -> {torch.xpu.get_device_properties(i).name}') "
    "for i in range(torch.xpu.device_count())]\"",
    shell=True, capture_output=True, text=True
)
print(r2.stdout)

If your environment passes all checks above, proceed to
[Allreduce -- The TP Decode Hot Path](03a_allreduce_walkthrough.ipynb).